# Planar truss GNN: axial force and static action

This notebook trains an edge-level PyTorch Geometric model on Pratt, Howe, Warren, K-truss, and Fink structures with variable panel counts.

The learned target is the member axial force. Cross-section and material properties are not model inputs; the dataset assumes nominal constant values.

In [ ]:
from pathlib import Path
import copy
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch.nn import functional as F
from torch.utils.data import Sampler
from torch_geometric.loader import DataLoader

from dataset import TrussDataset, grouped_split_indices, random_split_indices
from model import TrussAxialForceGNN

SEED = 42
DATA_ROOT = Path("data")

TYPOLOGIES = ["Pratt"]
MAX_SEEDS_PER_PANEL = 2
MAX_DESIGNS_PER_SEED = 500

BATCH_SIZE = 256
HIDDEN_DIM = 64
NUM_LAYERS = 4
DROPOUT = 0.05
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-5
MAX_EPOCHS = 100
VALIDATE_EVERY = 5
PATIENCE_CHECKS = 6

NUM_WORKERS = 0
CPU_THREADS = max(1, os.cpu_count() or 1)
torch.set_num_threads(CPU_THREADS)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("PyTorch CPU threads:", torch.get_num_threads())


## 1. Load and cache a selected subset

The dataset selects seed folders and designs **before opening JSON files**. Each selection configuration receives its own processed pickle, so changing the limits creates a separate cache automatically.

For the fast-development preset below, at most two seeds are selected for every Pratt panel count and at most 500 designs are selected from each seed folder.


In [ ]:
dataset = TrussDataset(
    root=DATA_ROOT,
    cache_name="trusses_fast",
    typologies=TYPOLOGIES,
    max_seeds_per_panel=MAX_SEEDS_PER_PANEL,
    max_designs_per_seed=MAX_DESIGNS_PER_SEED,
    sampling_seed=SEED,
    force_reload=False,
)

print(f"Graphs: {len(dataset):,}")
print("Selection:", dataset.metadata["selection"])
print("Typologies:", dataset.metadata["typologies"])
print("Node features:", dataset.metadata["node_features"])
print("Directed edge features:", dataset.metadata["directed_edge_features"])
print("Member features:", dataset.metadata["member_features"])
print("Processed cache:", dataset.processed_paths[0])

if len(dataset) < 3:
    raise RuntimeError("At least three selected graphs are required for training.")


In [ ]:
sample = dataset[0]
print(sample)
print("Nodes:", sample.num_nodes)
print("Physical members:", sample.member_index.shape[1])
print("Directed message-passing edges:", sample.edge_index.shape[1])
print("Applied-load scale:", float(sample.load_scale))
print("True static action:", float(sample.static_action))
print("First five normalized targets:", sample.y[:5].tolist())

## 2. Split and construct faster loaders

The train sampler visits only `TRAIN_GRAPHS_PER_EPOCH` graphs per epoch and draws a new subset on every pass. Validation uses a fixed smaller subset during model selection; final metrics still use the complete test split.

Training and validation loaders exclude tensors that are not needed by the loss, reducing collation and memory-copy overhead.


In [ ]:
train_idx, val_idx, test_idx = grouped_split_indices(
    dataset,
    train_fraction=0.80,
    val_fraction=0.10,
    seed=SEED,
)

#train_idx, val_idx, test_idx = random_split_indices(
#    len(dataset),
#    train_fraction=0.04,
#    val_fraction=0.10,
#    seed=SEED,
#)

train_set = dataset[train_idx]
val_set = dataset[val_idx]
test_set = dataset[test_idx]

In [ ]:
def fixed_subset(dataset_subset, max_graphs, seed):
    if max_graphs is None or len(dataset_subset) <= max_graphs:
        return dataset_subset
    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(len(dataset_subset), generator=generator)[:max_graphs]
    return dataset_subset[indices.tolist()]


class EpochSubsetSampler(Sampler):
    """Draw a new subset without replacement every epoch."""

    def __init__(self, dataset_size, num_samples, seed=42):
        self.dataset_size = int(dataset_size)
        self.num_samples = min(int(num_samples), self.dataset_size)
        self.generator = torch.Generator().manual_seed(seed)

    def __iter__(self):
        indices = torch.randperm(self.dataset_size, generator=self.generator)
        return iter(indices[: self.num_samples].tolist())

    def __len__(self):
        return self.num_samples


# None of these fields are needed for normalized axial-force training.
TRAIN_EXCLUDE_KEYS = [
    "pos",
    "pos_raw",
    "axial_force",
    "load_scale",
    "member_length",
    "static_action",
    "family_id",
    "group_id",
    "panels",
    "seed",
    "graph_id",
    "num_members",
]

worker_kwargs = {"num_workers": NUM_WORKERS}
if NUM_WORKERS > 0:
    worker_kwargs.update(
        persistent_workers=True,
        prefetch_factor=2,
    )



In [ ]:
USE_SUBSET = False  # toggle full-dataset vs epoch-subset training

if USE_SUBSET:
    # A new random subset of the training split is visited each epoch.
    TRAIN_GRAPHS_PER_EPOCH = 10_000
    VAL_GRAPHS = 2_000

    val_fast_set = fixed_subset(val_set, VAL_GRAPHS, SEED + 1)
    train_sampler = EpochSubsetSampler(
        len(train_set),
        TRAIN_GRAPHS_PER_EPOCH,
        seed=SEED + 2,
    )

    train_loader = DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        sampler=train_sampler,
        shuffle=False,
        drop_last=True,
        exclude_keys=TRAIN_EXCLUDE_KEYS,
        pin_memory=torch.cuda.is_available(),
        **worker_kwargs,
    )
    val_loader = DataLoader(
        val_fast_set,
        batch_size=BATCH_SIZE,
        shuffle=False,
        exclude_keys=TRAIN_EXCLUDE_KEYS,
        pin_memory=torch.cuda.is_available(),
        **worker_kwargs,
    )

    print(f"Full train split: {len(train_set):,}")
    print(f"Graphs visited per epoch: {len(train_sampler):,}")
    print(f"Validation subset: {len(val_fast_set):,} / {len(val_set):,}")

else:
    train_loader = DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        shuffle=True,
        exclude_keys=TRAIN_EXCLUDE_KEYS,
        pin_memory=torch.cuda.is_available(),
        **worker_kwargs,
    )
    val_loader = DataLoader(
        val_set,
        batch_size=BATCH_SIZE,
        shuffle=False,
        exclude_keys=TRAIN_EXCLUDE_KEYS,
        pin_memory=torch.cuda.is_available(),
        **worker_kwargs,
    )

    print(f"Full train split: {len(train_set):,}")
    print(f"Full validation split: {len(val_set):,}")

test_loader = DataLoader(
    test_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    exclude_keys=TRAIN_EXCLUDE_KEYS,
    pin_memory=torch.cuda.is_available(),
    **worker_kwargs,
)

print(f"Full test split: {len(test_set):,}")
print(f"Training batches per epoch: {len(train_loader):,}")

## 3. Model and graph-balanced loss

Each graph can contain a different number of members. The loss first averages member errors inside each graph and then averages across graphs, preventing larger trusses from dominating solely because they have more edges.

In [ ]:
model = TrussAxialForceGNN(
    node_dim=sample.x.shape[-1],
    directed_edge_dim=sample.edge_attr.shape[-1],
    member_dim=sample.member_attr.shape[-1],
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
)

num_parameters = sum(parameter.numel() for parameter in model.parameters())
print(f"Trainable parameters: {num_parameters:,}")


In [ ]:
def graph_balanced_smooth_l1(prediction, target, member_batch, num_graphs):
    per_member = F.smooth_l1_loss(prediction, target, reduction="none")
    graph_sum = torch.zeros(num_graphs, device=prediction.device, dtype=prediction.dtype)
    graph_sum.index_add_(0, member_batch, per_member)
    graph_count = torch.bincount(member_batch, minlength=num_graphs).clamp_min(1)
    return (graph_sum / graph_count).mean()


def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    total_graphs = 0

    for batch in loader:
        batch = batch.to(device)
        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            # Physical forces and static action are skipped; not needed
            #  the loss function.
            output = model(batch, compute_static_action=False)
            loss = graph_balanced_smooth_l1(
                output["axial_norm"],
                batch.y,
                output["member_batch"],
                batch.num_graphs,
            )
            if training:
                loss.backward()
                optimizer.step()

        total_loss += float(loss) * batch.num_graphs
        total_graphs += batch.num_graphs

    return total_loss / max(total_graphs, 1)

## 4. Training

In [ ]:
history = {"train": [], "val": [], "lr": [], "seconds": []}
best_state = None
best_val = float("inf")
patience_counter = 0

for epoch in range(1, MAX_EPOCHS + 1):
    start = time.perf_counter()
    train_loss = run_epoch(train_loader, training=True)

    should_validate = (
        epoch == 1
        or epoch % VALIDATE_EVERY == 0
        or epoch == MAX_EPOCHS
    )

    if should_validate:
        val_loss = run_epoch(val_loader, training=False)
        scheduler.step(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
    else:
        val_loss = float("nan")

    elapsed = time.perf_counter() - start
    learning_rate = optimizer.param_groups[0]["lr"]
    history["train"].append(train_loss)
    history["val"].append(val_loss)
    history["lr"].append(learning_rate)
    history["seconds"].append(elapsed)

    if should_validate or epoch % 5 == 0:
        val_text = f"{val_loss:.6f}" if should_validate else "skipped"
        print(
            f"Epoch {epoch:03d} | train={train_loss:.6f} | "
            f"val={val_text} | lr={learning_rate:.2e} | {elapsed:.1f}s"
        )

    if should_validate and patience_counter >= PATIENCE_CHECKS:
        print(
            f"Early stopping at epoch {epoch}; "
            f"best validation loss={best_val:.6f}"
        )
        break

if best_state is None:
    raise RuntimeError("Training did not produce a validated model state.")
model.load_state_dict(best_state)

In [ ]:
epochs = np.arange(1, len(history["train"]) + 1)
validation = np.asarray(history["val"], dtype=float)
validated = np.isfinite(validation)

plt.figure(figsize=(7, 4.5))
plt.plot(epochs, history["train"], label="Train")
plt.plot(epochs[validated], validation[validated], marker="o", label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Graph-balanced Smooth L1 loss")
plt.title("Training history")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Mean epoch time: {np.mean(history['seconds']):.1f}s")


## 5. Collect physical-force and static-action predictions

In [ ]:
@torch.no_grad()
def collect_predictions(loader):
    model.eval()
    axial_true = []
    axial_pred = []
    static_true = []
    static_pred = []

    for batch in loader:
        batch = batch.to(device)
        output = model(batch)
        axial_true.append(batch.axial_force.detach().cpu())
        axial_pred.append(output["axial_force"].detach().cpu())
        static_true.append(batch.static_action.view(-1).detach().cpu())
        static_pred.append(output["static_action"].detach().cpu())

    return {
        "axial_true": torch.cat(axial_true).numpy(),
        "axial_pred": torch.cat(axial_pred).numpy(),
        "static_true": torch.cat(static_true).numpy(),
        "static_pred": torch.cat(static_pred).numpy(),
    }

predictions = collect_predictions(test_loader)

In [ ]:
def regression_metrics(y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    metrics = {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse,
        "R2": r2_score(y_true, y_pred) if len(y_true) > 1 else float("nan"),
    }
    nonzero = np.abs(y_true) > 1e-8
    if np.any(nonzero):
        metrics["MAPE_nonzero_%"] = np.mean(
            np.abs((y_pred[nonzero] - y_true[nonzero]) / y_true[nonzero])
        ) * 100.0
    return metrics

axial_metrics = regression_metrics(predictions["axial_true"], predictions["axial_pred"])
static_metrics = regression_metrics(predictions["static_true"], predictions["static_pred"])
sign_accuracy = np.mean(
    np.sign(predictions["axial_true"]) == np.sign(predictions["axial_pred"])
)

print("Axial-force metrics")
for name, value in axial_metrics.items():
    print(f"  {name}: {value:.6g}")
print(f"  tension/compression sign accuracy: {100.0 * sign_accuracy:.2f}%")

print("\nStatic-action metrics")
for name, value in static_metrics.items():
    print(f"  {name}: {value:.6g}")


## 6. Axial-force performance

In [ ]:
rng = np.random.default_rng(SEED)
num_edges = len(predictions["axial_true"])
max_points = 40_000
if num_edges > max_points:
    shown = rng.choice(num_edges, size=max_points, replace=False)
else:
    shown = np.arange(num_edges)

true_force = predictions["axial_true"][shown]
pred_force = predictions["axial_pred"][shown]
limit = max(np.max(np.abs(true_force)), np.max(np.abs(pred_force)))

plt.figure(figsize=(6, 6))
plt.scatter(true_force, pred_force, s=8, alpha=0.35)
plt.plot([-limit, limit], [-limit, limit], linestyle="--", linewidth=1.5)
plt.xlabel("True axial force")
plt.ylabel("Predicted axial force")
plt.title(f"Member axial-force parity, test set (R²={axial_metrics['R2']:.3f})")
plt.axis("equal")
plt.xlim(-limit, limit)
plt.ylim(-limit, limit)
plt.tight_layout()
plt.show()

In [ ]:
axial_error = predictions["axial_pred"] - predictions["axial_true"]

plt.figure(figsize=(7, 4.5))
plt.hist(axial_error, bins=80)
plt.xlabel("Predicted - true axial force")
plt.ylabel("Member count")
plt.title("Axial-force error distribution")
plt.tight_layout()
plt.show()

## 7. Static-action performance

Static action is not independently predicted. It is calculated from the GNN's physical member-force predictions and the exact member lengths. This directly tests whether local force errors preserve the global structural quantity.

In [ ]:
true_static = predictions["static_true"]
pred_static = predictions["static_pred"]
low = min(true_static.min(), pred_static.min())
high = max(true_static.max(), pred_static.max())

plt.figure(figsize=(6, 6))
plt.scatter(true_static, pred_static, s=18, alpha=0.55)
plt.plot([low, high], [low, high], linestyle="--", linewidth=1.5)
plt.xlabel("True static action")
plt.ylabel("Static action from predicted axial forces")
plt.title(f"Static-action parity, test set (R²={static_metrics['R2']:.3f})")
plt.axis("equal")
plt.xlim(low, high)
plt.ylim(low, high)
plt.tight_layout()
plt.show()

In [ ]:
relative_static_error = (
    (predictions["static_pred"] - predictions["static_true"])
    / np.maximum(np.abs(predictions["static_true"]), 1e-8)
) * 100.0

plt.figure(figsize=(7, 4.5))
plt.hist(relative_static_error, bins=60)
plt.xlabel("Static-action relative error (%)")
plt.ylabel("Structure count")
plt.title("Static-action error distribution")
plt.tight_layout()
plt.show()

## 8. Visualize one test truss

In [ ]:
from matplotlib.collections import LineCollection
from matplotlib.colors import TwoSlopeNorm


def predict_single_graph(graph):
    model.eval()
    graph_device = graph.to(device)
    with torch.no_grad():
        output = model(graph_device)
    return output["axial_force"].detach().cpu().numpy()


def plot_member_field(graph, values, title, norm, cmap="bwr_r"):
    positions = graph.pos_raw.cpu().numpy()
    member_index = graph.member_index.cpu().numpy()
    segments = [positions[member_index[:, idx]] for idx in range(member_index.shape[1])]

    collection = LineCollection(
        segments,
        linewidths=3,
        cmap=cmap,
        norm=norm,
    )
    collection.set_array(np.asarray(values))

    figure, axis = plt.subplots(figsize=(10, 4))
    axis.add_collection(collection)
    axis.scatter(positions[:, 0], positions[:, 1], s=18, color="black")
    axis.autoscale()
    axis.set_aspect("equal")
    axis.set_xlabel("X")
    axis.set_ylabel("Z")
    axis.set_title(title)
    figure.colorbar(collection, ax=axis, label="Axial force")
    figure.tight_layout()
    plt.show()


example = test_set[0]
true_values = example.axial_force.numpy()
pred_values = predict_single_graph(example)

# shared symmetric scale
shared_bound = max(
    float(np.max(np.abs(true_values))),
    float(np.max(np.abs(pred_values))),
    1e-8,
)

shared_norm = TwoSlopeNorm(
    vmin=-shared_bound,
    vcenter=0.0,
    vmax=shared_bound,
)

plot_member_field(
    example,
    true_values,
    "True member axial forces",
    norm=shared_norm,
    cmap="bwr_r",
)

plot_member_field(
    example,
    pred_values,
    "Predicted member axial forces",
    norm=shared_norm,
    cmap="bwr_r",
)

## 9. Save the trained model

In [ ]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "model_config": {
        "node_dim": sample.x.shape[-1],
        "directed_edge_dim": sample.edge_attr.shape[-1],
        "member_dim": sample.member_attr.shape[-1],
        "hidden_dim": HIDDEN_DIM,
        "num_layers": NUM_LAYERS,
        "dropout": DROPOUT,
    },
    "dataset_metadata": dataset.metadata,
    "best_validation_loss": best_val,
    "seed": SEED,
}

checkpoint_path = Path("checkpoints") / "truss_axial_force_gnn.pt"
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(checkpoint, checkpoint_path)
print("Saved:", checkpoint_path)